# _This notebook focuses on the Slowly Changing Dimension Type 2 (SCD2) implementation extracted from a larger end-to-end data engineering project_

_It is part from the **➡️ ["PySpark Medallion Architecture: E2E Pipeline"](https://github.com/tkocza/PySpark-projects/blob/main/Pyspark%20E2E%20medallion%20architecture/pyspark-medallion-architecture-e2e-pipeline.ipynb)** project, which demonstrates a complete Medallion Architecture (Bronze → Silver → Gold) ending with a dimensional model (star schema)._

<div align="center">

# **Goal**

</div>


### This notebook demonstrates how to implement a **production-ready Slowly Changing Dimension Type 2 (SCD2)** process in PySpark.


The focus is not only on making SCD2 work, but on applying **best practices commonly used in modern data warehouses**, including:

- change detection
- historical versioning
- surrogate key management
- current record identification
- effective date ranges
- handling unknown and invalid references

### The objective is to build a dimension that is reliable, scalable, and ready for analytical workloads.



<div align="center">
    
# **What are Slowly Changing Dimensions?**
## Slowly Changing Dimensions (SCD)

</div>

Dimensions often contain attributes that change over time (customer address, employee department, product category, etc.). Slowly Changing Dimensions define **how these changes should be handled**.

| Type | Description |
|------|-------------|
| **Type 0** | Never update the attribute. |
| **Type 1** | Overwrite the existing value (no history). |
| **Type 2** | Insert a new row for every change and preserve full history. |
| **Type 3** | Store previous value in additional columns (limited history). |
| **Type 4** | Keep current data in the dimension and historical data in a separate history table. |
| **Type 5,6** | Hybrid approaches combining multiple SCD techniques. |

This notebook focuses on **Type 2**, the most commonly used approach in enterprise data warehouses.

<div align="center">

# SCD Types at a Glance

<img src="pics/scd_types.png">

</div>

<div align="center">

# Why is SCD2 the industry standard?

</div>

### SCD2 preserves every historical version of a record instead of overwriting existing data.

This enables:

- complete audit trail of business changes
- point-in-time reporting
- reproducible historical analyses
- accurate joins between facts and dimensions using effective dates
- regulatory and compliance reporting

Instead of updating a row, the previous version is closed (`valid_to`) and a new version is inserted (`valid_from`), ensuring the full business history remains available.

```text
Customer A

Before:
┌────┬─────────┬────────────┬──────────┐
│SK  │ City    │ valid_from │ valid_to │
├────┼─────────┼────────────┼──────────┤
│101 │ London  │2023-01-01  │3000-12-31│
└────┴─────────┴────────────┴──────────┘

After city change:

┌────┬─────────┬────────────┬──────────┐
│101 │ London  │2023-01-01  │2024-06-30│
│102 │ Paris   │2024-07-01  │3000-12-31│
└────┴─────────┴────────────┴──────────┘
```

## The implementation shown below follows the same sequence of operations commonly used in production ETL pipelines, making it scalable and easy to maintain.

<div align="center">

# Process
<img src="pics/scd_process.png" width="600">
</div>

# INTRODUCTION

## Load libraries

In [1]:
import os
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

import warnings

warnings.filterwarnings("ignore")

## Environment variables

In [2]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/organizations/usdot/flight-delays/airports.csv
/kaggle/input/datasets/organizations/usdot/flight-delays/airlines.csv
/kaggle/input/datasets/organizations/usdot/flight-delays/flights.csv


In [3]:
ENV_file_name_airports = 'airports.csv'

ENV_raw_data_path = '/kaggle/input/datasets/organizations/usdot/flight-delays'

## Start spark session

In [4]:
spark = SparkSession.builder.appName("SCD2_dim").getOrCreate()
spark.sparkContext.setLogLevel("ERROR") # show only critical message
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/15 11:01:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# BRONZE layer

## Create schema based on documentation

<div align='center'>
    
### Airport Reference Dataset Schema

</div>
<br><br>

| Column | Data Type | Notes / Comments |
|---|---|---|
| IATA_CODE | StringType | Unique airport IATA code (3-letter identifier) |
| AIRPORT | StringType | Full airport name |
| CITY | StringType | City where airport is located |
| STATE | StringType | State / region (mainly US-based dataset) |
| COUNTRY | StringType | Country of airport |
| LATITUDE | DoubleType | Geographic latitude coordinate |
| LONGITUDE | DoubleType | Geographic longitude coordinate |

In [5]:
schema_airports = StructType([
    StructField("IATA_CODE", StringType(), True, {"comment": "Unique airport IATA code (3-letter)"}),
    StructField("AIRPORT", StringType(), True, {"comment": "Full airport name"}),
    StructField("CITY", StringType(), True, {"comment": "City of airport location"}),
    StructField("STATE", StringType(), True, {"comment": "State or region"}),
    StructField("COUNTRY", StringType(), True, {"comment": "Country"}),
    StructField("LATITUDE", DoubleType(), True, {"comment": "Latitude coordinate"}),
    StructField("LONGITUDE", DoubleType(), True, {"comment": "Longitude coordinate"})
])

## Load data

In [6]:
df_airports_bronze = spark.read.csv(os.path.join(ENV_raw_data_path, ENV_file_name_airports), header=True, inferSchema=True)

## Check data

In [7]:
display(df_airports_bronze.limit(10).toPandas())
df_airports_bronze.columns

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447
5,ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018
6,ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052
7,ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862
8,ACY,Atlantic City International Airport,Atlantic City,NJ,USA,39.45758,-74.57717
9,ADK,Adak Airport,Adak,AK,USA,51.87796,-176.64603


['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE', 'LONGITUDE']

## Make columns lowercase

In [8]:
def to_lowercase_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.lower())
    return df

df_airports_bronze = to_lowercase_columns(df_airports_bronze)

## Add technical columns for maintenance

In [9]:
df_airports_bronze = df_airports_bronze.withColumn('load_date', F.current_timestamp())
df_airports_bronze = df_airports_bronze.withColumn('source_file', F.lit(ENV_file_name_airports))

# SILVER layer

In [10]:
excluded_columns = ["load_date", "source_file"]
df_airports_silver = df_airports_bronze.drop(*excluded_columns)

## Rename business key column to make it more informative

In [11]:
def build_row_hash(cols):
    record_hash = F.sha2(F.concat_ws("||",*[F.concat(F.lit(c), F.lit("="), F.coalesce(F.col(c).cast("string"), F.lit("NULL"))) for c in cols]), 256)
    return record_hash

## Create a hash function for attributes
A row hash is used instead of comparing every business column individually. This simplifies the comparison logic and scales better as the dimension grows.

#### Row Hash Strategy
Each attribute is encoded as:
`column_name=value`

Including column names makes the hash **self-describing** and eliminates ambiguity caused by missing (`NULL`) values or changes in column order. This results in more robust and deterministic change detection, making the hashing strategy safer for production ETL pipelines.


In [12]:
rename_cols_map_airports = {
    'iata_code': 'airport_code'
}
df_airports_silver = df_airports_silver.withColumnsRenamed(rename_cols_map_airports)

## Generate hash for rows

In [13]:
# exclude identifier columns
exclude_key = ["airport_code"]
cols_to_hash = [c for c in df_airports_silver.columns if c not in exclude_key]
df_airports_silver = df_airports_silver.withColumn("row_hash", build_row_hash(cols_to_hash))

# GOLD layer

In [14]:
# make safe copy, not shallow copy
df_airports_gold = df_airports_silver.select(*df_airports_silver.columns)

## Initial Dimension Load

(Optional step) Execute only when required by business requirements.  
This step creates the initial dimension and is executed only once, before incremental SCD2 loads begin

**The default values 1900-01-01 for valid_from and 3000-12-31 for valid_to are commonly used in data warehouses. However, these values may differ depending on the project's business or technical requirements.**

In [15]:
# SCD2 simulation; global ordering acceptable despite Spark warning about missing partition (intentional single-partition design)
window = Window.orderBy("airport_code")

initial_dim = df_airports_gold.withColumn("airport_id", F.row_number().over(window))

dim_airports = (initial_dim
    #.withColumn("valid_from", F.current_timestamp()) # depends on approach
    .withColumn("valid_from", F.lit('1900-01-01').cast("timestamp"))
    .withColumn("valid_to", F.lit('3000-12-31').cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

# change the order of column to put at the begining id, hash, valid_from, valid_to, is_current
# technical columns are first because in case of dimension extention they will not be in the middle of table
dim_airports = dim_airports.select(
    "airport_id",
    "row_hash",
    "valid_from",
    "valid_to",
    "is_current",
    "airport_code",
    "airport",
    "city",
    "state",
    "country",
    "latitude",
    "longitude"
)

<div align="center">

# Why create special records?



### A production dimension should always contain predefined surrogate keys for exceptional cases.

</div>

| Surrogate Key | Record | Purpose |
|--------------:|--------|---------|
| **-1** | UNKNOWN | Business key is missing or does not exist in the source system. |
| **-999** | QUARANTINED | Invalid or corrupted data requiring investigation. |

Using dedicated records instead of `NULL` foreign keys provides several advantages:

- referential integrity is preserved
- fact loading never fails because of missing dimension members
- data quality issues are explicitly identified
- downstream reporting becomes simpler and more consistent

This approach is widely adopted in enterprise data warehouse implementations.

In [16]:
# the order of column is important
unknown_airport = spark.createDataFrame(
    [
        Row(
            airport_id=-1,
            row_hash=None,
            valid_from=None,
            valid_to=None,
            is_current=True,
            airport_code="UNKNOWN",
            airport="Unknown Airport",
            city="Unknown",
            state="Unknown",
            country="Unknown",
            latitude=None,
            longitude=None
        )
    ],
    schema=dim_airports.schema
)

quarantined_airport = spark.createDataFrame(
    [
        Row(
            airport_id=-999,
            row_hash=None,
            valid_from=None,
            valid_to=None,
            is_current=True,
            airport_code="QUARANTINED",
            airport="Missing Airport",
            city="Unknown",
            state="Unknown",
            country="Unknown",
            latitude=None,
            longitude=None
        )
    ],
    schema=dim_airports.schema
)
#dim_airports = unknown_airport.unionByName(quarantined_airport).unionByName(dim_airports)
if (dim_airports.filter(F.col("airport_id") == -1).count() == 0) & (dim_airports.filter(F.col("airport_id") == -999).count() == 0):
    dim_airports = unknown_airport.unionByName(quarantined_airport).unionByName(dim_airports)
else:
    print("Records already existed")

## Check initial dimension

In [17]:
'''
dim_airports.limit(10).toPandas()
doesn't work because converting to pandas changes timestamp columns to datetime64[ns]
which max is Timestamp('2262-04-11 23:47:16.854775807')
and valid_to is 3000-01-01
'''
(
    dim_airports.select(
        [F.col(c).cast("string").alias(c)
         if c in ["valid_to"] else F.col(c)
         for c in dim_airports.columns])
    .limit(10).toPandas()
)

,airport_id,row_hash,valid_from,valid_to,is_current,airport_code,airport,city,state,country,latitude,longitude
0,-1,None,NaT,None,True,UNKNOWN,Unknown Airport,Unknown,Unknown,Unknown,NaN,NaN
1,-999,None,NaT,None,True,QUARANTINED,Missing Airport,Unknown,Unknown,Unknown,NaN,NaN
2,1,3ee4e61e6bb79f4087692689e042b98c27ac51e0e84764...,1900-01-01,3000-12-31 00:00:00,True,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
3,2,a10b7984c849486313cb38bb2b9ca8316129016971f9cf...,1900-01-01,3000-12-31 00:00:00,True,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
4,3,0d3fea02e1c8485a39c799bc97ef24a4df9c03a36285a7...,1900-01-01,3000-12-31 00:00:00,True,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
5,4,67452d0ee858e3883654ead2e9c1e438a458685351b206...,1900-01-01,3000-12-31 00:00:00,True,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
6,5,8ac8c40bfcf8c19d55438d7b52344709e321a10f13359f...,1900-01-01,3000-12-31 00:00:00,True,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447
7,6,73384b854259be92e1d3be06b417e0eb88a45ce309c379...,1900-01-01,3000-12-31 00:00:00,True,ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018
8,7,ba7c53e898311de7b1de834e44daf72dd8c1e8ceee5779...,1900-01-01,3000-12-31 00:00:00,True,ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052
9,8,af4c2ce700dc65806ccc810071590bd061e955a461bf84...,1900-01-01,3000-12-31 00:00:00,True,ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862


## Dimension Load

The load process identifies new, changed, and deleted records from the source system and applies the appropriate SCD Type 2 actions.

## Simulate incoming new data
- ABE - changed record  
- ABQ - deleted record (removed from source)

In [18]:
df_new_batch = (df_airports_silver
                .drop("row_hash")
                .filter(F.col("airport_code") != "ABQ")
                .withColumn("airport", 
                            F.when(F.col("airport_code") == "ABE", F.lit("Lehigh Valley International Airport New"))
                            .otherwise(F.col("airport")))
                )
# build hash for new batch
exclude_columns = ["airport_code"]
cols_to_hash = [c for c in df_new_batch.columns if c not in exclude_columns]
df_new_batch = df_new_batch.withColumn("row_hash",build_row_hash(cols_to_hash))

## Take last version of records and exclude -1 and -999 records

In [19]:
current_dim = dim_airports.filter(F.col("is_current") & (~F.col("airport_id").isin(-1, -999)))

## Detect SCD2 changes and put proper label
We compare the latest active dimension records with the incoming source data using the row hash. Based on the comparison, each business key is classified as **NEW, CHANGED, UNCHANGED, or DELETED**

In [20]:
records_with_status = (
    df_new_batch.alias("new")
    .join(current_dim.alias("old"), "airport_code", "full")
    .withColumn("airport_code_key", F.coalesce(F.col("new.airport_code"), F.col("old.airport_code")))
    .withColumn(
        "record_status",
        F.when(F.col("old.airport_code").isNull(), "NEW")
         .when(F.col("new.airport_code").isNull(), "DELETED")
         .when(F.col("new.row_hash") != F.col("old.row_hash"), "CHANGED")
         .otherwise("UNCHANGED")
    )
)

## Find active records that must be closed (last records for "changed" and "deleted")
Only the current active version (is_current = true) can be closed. Historical records remain untouched to preserve the full change history.

In [21]:
keys_to_expire = (
    records_with_status
    .filter(F.col("record_status").isin("CHANGED", "DELETED"))
    .select(F.col("airport_code_key").alias("airport_code"))
    .distinct()
)

## Close old versions
Changed and deleted records are expired by updating their valid_to timestamp and setting is_current to false. This preserves history while ensuring that only one version remains active.

In [22]:
df_expired = (
    dim_airports
    .filter(F.col("is_current"))
    .join(keys_to_expire, "airport_code")
    .withColumn("valid_to", F.current_timestamp())
    .withColumn("is_current", F.lit(False))
)

## Keep unchanged records and historical records
These records require no modification and can be carried over directly to the next dimension version. This avoids unnecessary updates and keeps the pipeline efficient.

In [23]:
df_unchanged = dim_airports.join(keys_to_expire, "airport_code", "left_anti")

## Extract rows for new version ("NEW" and "CHANGED")
New business keys and changed records will become new active versions in the dimension. At this stage we prepare only the rows that need to be inserted.

In [24]:
records_to_insert = (
    records_with_status
    .filter(F.col("record_status").isin("NEW", "CHANGED"))
    .select("new.*")
)

## Generate surrogate keys
Every inserted dimension row receives a new surrogate key, even if the business key already exists. This allows multiple historical versions of the same business entity to coexist.

In [25]:
max_id = dim_airports.agg(F.max("airport_id").alias("max_id")).first()["max_id"] or 0

## Prepare new dimension records
The final insert dataset combines completely new business keys with new versions created for changed records. All of them become active dimension records.

In [26]:
df_insert = (
    records_to_insert
    .withColumn("airport_id", F.row_number().over(Window.orderBy("airport_code")) + F.lit(max_id))
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit("3000-12-31").cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

## Rebuild dimension
The updated dimension is created by combining unchanged records, historical records, closed versions, and newly inserted active versions. The result represents the complete SCD Type 2 state after the load.

In [27]:
dim_airports_updated = (
    df_unchanged
    .unionByName(df_expired)
    .unionByName(df_insert)
)

## Final validation

In [28]:

print("------------------------------------")
print("Final dimension counts:")
print("------------------------------------")
display(dim_airports_updated.groupBy("is_current").count().limit(30).toPandas())

print("------------------------------------")
print("Records by status:")
print("------------------------------------")
display(records_with_status.groupBy("record_status").count().limit(30).toPandas())

display(
    dim_airports.orderBy("airport_code", "valid_from").select(
        [F.col(c).cast("string").alias(c)
         if c in ["valid_to"] else F.col(c)
         for c in dim_airports.columns])
    .limit(10).toPandas()
)

------------------------------------
Final dimension counts:
------------------------------------


,is_current,count
0,True,323
1,False,2


------------------------------------
Records by status:
------------------------------------


,record_status,count
0,CHANGED,1
1,DELETED,1
2,UNCHANGED,320


,airport_id,row_hash,valid_from,valid_to,is_current,airport_code,airport,city,state,country,latitude,longitude
0,1,3ee4e61e6bb79f4087692689e042b98c27ac51e0e84764...,1900-01-01,3000-12-31 00:00:00,True,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,2,a10b7984c849486313cb38bb2b9ca8316129016971f9cf...,1900-01-01,3000-12-31 00:00:00,True,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,3,0d3fea02e1c8485a39c799bc97ef24a4df9c03a36285a7...,1900-01-01,3000-12-31 00:00:00,True,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,4,67452d0ee858e3883654ead2e9c1e438a458685351b206...,1900-01-01,3000-12-31 00:00:00,True,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,5,8ac8c40bfcf8c19d55438d7b52344709e321a10f13359f...,1900-01-01,3000-12-31 00:00:00,True,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447
5,6,73384b854259be92e1d3be06b417e0eb88a45ce309c379...,1900-01-01,3000-12-31 00:00:00,True,ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018
6,7,ba7c53e898311de7b1de834e44daf72dd8c1e8ceee5779...,1900-01-01,3000-12-31 00:00:00,True,ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052
7,8,af4c2ce700dc65806ccc810071590bd061e955a461bf84...,1900-01-01,3000-12-31 00:00:00,True,ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862
8,9,dc0c550177071b0ecce3c106c3dea10cd038c83727e300...,1900-01-01,3000-12-31 00:00:00,True,ACY,Atlantic City International Airport,Atlantic City,NJ,USA,39.45758,-74.57717
9,10,226e3bc2e73bf51e29b5599ca1a0e18f7cfc27e58f3039...,1900-01-01,3000-12-31 00:00:00,True,ADK,Adak Airport,Adak,AK,USA,51.87796,-176.64603


<div align = 'center'>

# Summary
### This implementation preserves complete business history while keeping only one active version of each business key. The same pattern can be applied to most dimensions in a modern data warehouse.

</div>